In [1]:
import pandas as pd


In [2]:
rfm = pd.read_csv('../data/rfm_customer_data.csv')
rfm.head()

,customer_unique_id,order_purchase_timestamp,recency,frequency,monetary,R_score,M_score,F_score,RFM_score
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,161,1,141.90,4,3,1,413
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,164,1,27.19,4,1,1,411
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,586,1,86.22,1,2,1,112
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,370,1,43.62,2,1,1,211
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,337,1,196.89,2,4,1,214


In [3]:
rfm['churned'] = (rfm['recency'] > 180).astype(int)
rfm['churned'].value_counts(normalize=True)

churned
1    0.711289
0    0.288711
Name: proportion, dtype: float64

In [4]:
orders = pd.read_csv('../data/olist_orders_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

orders_payments = orders.merge(payments, on='order_id', how='left')
orders_full = orders_payments.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')

In [5]:
extra_features = orders_full.groupby('customer_unique_id').agg(
    avg_order_value=('payment_value', 'mean'),
    avg_installments=('payment_installments', 'mean')
).reset_index()

rfm = rfm.merge(extra_features, on='customer_unique_id', how='left')
rfm.head()

,customer_unique_id,order_purchase_timestamp,recency,frequency,monetary,R_score,M_score,F_score,RFM_score,churned,avg_order_value,avg_installments
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,161,1,141.90,4,3,1,413,0,141.90,8.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,164,1,27.19,4,1,1,411,0,27.19,1.0
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,586,1,86.22,1,2,1,112,1,86.22,8.0
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,370,1,43.62,2,1,1,211,1,43.62,4.0
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,337,1,196.89,2,4,1,214,1,196.89,6.0


In [6]:
feature_cols = ['recency', 'frequency', 'monetary', 'avg_order_value', 'avg_installments']

X = rfm[feature_cols]
y = rfm['churned']

X.isnull().sum()

recency             0
frequency           0
monetary            0
avg_order_value     1
avg_installments    1
dtype: int64

In [7]:
rfm[rfm['avg_order_value'].isnull()]

,customer_unique_id,order_purchase_timestamp,recency,frequency,monetary,R_score,M_score,F_score,RFM_score,churned,avg_order_value,avg_installments
49312,830d5b7aaa3b6f1e9ad63703bec97d23,2016-09-15 12:16:38,763,1,0.0,1,1,2,121,1,NaN,NaN


In [8]:
rfm = rfm.dropna(subset=['avg_order_value', 'avg_installments'])
X = rfm[feature_cols]
y = rfm['churned']

X.isnull().sum()

recency             0
frequency           0
monetary            0
avg_order_value     0
avg_installments    0
dtype: int64

In [9]:
feature_cols = ['frequency', 'monetary', 'avg_order_value', 'avg_installments']

X = rfm[feature_cols]
y = rfm['churned']

In [10]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = RandomForestClassifier(n_estimators=200,max_depth=20,class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

              precision    recall  f1-score   support

           0       0.41      0.65      0.51      5549
           1       0.81      0.63      0.71     13670

    accuracy                           0.63     19219
   macro avg       0.61      0.64      0.61     19219
weighted avg       0.70      0.63      0.65     19219

ROC-AUC: 0.6914804765892956


In [15]:
import joblib
joblib.dump(model, '../models/churn_model.pkl',compress=3)

['../models/churn_model.pkl']